# Challenge 2: entrenamiento y selección del modelo

En esta etapa debes construir pipelines reproducibles, comparar modelos y seleccionar un campeón sin utilizar el conjunto de test.

**Entradas:** `train.csv` y `data_contract.json`  
**Salidas:** `champion_model.joblib`, `training_metadata.json` y `cv_results.csv`

## Reglas

- El preprocesamiento debe estar dentro de cada `Pipeline`.
- La selección debe hacerse con validación cruzada.
- No cargues `test.csv` en este notebook.

In [1]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [2]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga de artefactos

In [3]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]

print(X_train.shape, y_train.shape)

(142, 13) (142,)


## 2. Pipeline de procesamiento

In [4]:
feature_columns = contract["features"]

# TODO: crea un preprocesador para árboles con imputación por mediana
# Los arboles parten cada variable con su propio umbral: escalar no cambia el arbol.
preprocesador_arbol = ColumnTransformer(
    transformers=[
        ("numericas", SimpleImputer(strategy="median"), feature_columns),
    ],
    remainder="drop",
)

# TODO: crea otro con imputación y StandardScaler
# KNN mide distancias y la regresion logistica converge por gradiente:
# las dos necesitan todas las variables en la misma escala.
preprocesador_escalado = ColumnTransformer(
    transformers=[
        (
            "numericas",
            Pipeline(
                steps=[
                    ("imputador", SimpleImputer(strategy="median")),
                    ("escalador", StandardScaler()),
                ]
            ),
            feature_columns,
        ),
    ],
    remainder="drop",
)

print("Preprocesador para arboles:")
print(preprocesador_arbol)
print("\nPreprocesador con escalado:")
print(preprocesador_escalado)


Preprocesador para arboles:
ColumnTransformer(transformers=[('numericas', SimpleImputer(strategy='median'),
                                 ['alcohol', 'malic_acid', 'ash',
                                  'alcalinity_of_ash', 'magnesium',
                                  'total_phenols', 'flavanoids',
                                  'nonflavanoid_phenols', 'proanthocyanins',
                                  'color_intensity', 'hue',
                                  'od280/od315_of_diluted_wines', 'proline'])])

Preprocesador con escalado:
ColumnTransformer(transformers=[('numericas',
                                 Pipeline(steps=[('imputador',
                                                  SimpleImputer(strategy='median')),
                                                 ('escalador',
                                                  StandardScaler())]),
                                 ['alcohol', 'malic_acid', 'ash',
                                  'alcalinity_of_ash'

**Pregunta:** ¿por qué el `StandardScaler` debe estar dentro del pipeline y no ajustarse antes de la validación cruzada?

> Porque ajustarlo antes provoca **fuga de datos** (*data leakage*). El `StandardScaler` no aplica una fórmula fija: aprende dos parámetros por columna —la media y la desviación estándar— y luego los usa para transformar. Si se ajusta sobre todo `X_train`, esos parámetros se calculan también con las filas que después harán de conjunto de validación en cada fold. Ese fold deja de ser "datos no vistos", porque información suya ya quedó incorporada en la transformación, y el accuracy de validación resulta optimista: mejor del que el modelo alcanzaría ante datos realmente nuevos.
>
> Al colocar el escalador dentro del `Pipeline`, `cross_validate` y `GridSearchCV` lo reajustan en cada iteración usando **solo** las filas de entrenamiento de ese fold, y aplican esa media y esa desviación al fold de validación sin recalcularlas. Así cada fold reproduce fielmente la situación de producción: transformar datos nuevos con las estadísticas aprendidas durante el entrenamiento. El mismo argumento vale para el `SimpleImputer`, que aprende la mediana de cada columna.
>
> Hay además una razón operativa: el pipeline encapsula preprocesamiento y modelo en un único objeto, de modo que al persistirlo con `joblib` y cargarlo en la etapa de inferencia es imposible olvidar un paso o aplicarlo en distinto orden. La secuencia que se validó es exactamente la que se despliega.


## 3. Modelos candidatos

In [5]:
# TODO: crea pipelines para DecisionTree, LogisticRegression y KNN
# El segundo paso se llama "clasificador" en los tres para poder referenciar
# sus hiperparametros como clasificador__<parametro> en el GridSearchCV.

models = {
    "decision_tree": Pipeline(
        steps=[
            ("preprocesamiento", clone(preprocesador_arbol)),
            ("clasificador", DecisionTreeClassifier(random_state=RANDOM_STATE)),
        ]
    ),
    "logistic_regression": Pipeline(
        steps=[
            ("preprocesamiento", clone(preprocesador_escalado)),
            ("clasificador", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
        ]
    ),
    "knn": Pipeline(
        steps=[
            ("preprocesamiento", clone(preprocesador_escalado)),
            ("clasificador", KNeighborsClassifier()),
        ]
    ),
}

for nombre, pipeline in models.items():
    escalado = "con escalado" if "escalador" in str(pipeline.named_steps["preprocesamiento"]) else "sin escalado"
    print(f"{nombre:22} {type(pipeline.named_steps['clasificador']).__name__:24} {escalado}")


decision_tree          DecisionTreeClassifier   sin escalado
logistic_regression    LogisticRegression       con escalado
knn                    KNeighborsClassifier     con escalado


## 4. Comparación inicial con validación cruzada

In [6]:
# TODO: usa StratifiedKFold de 5 folds y cross_validate
# Reporta accuracy promedio de train, accuracy promedio de validation y desviación estándar
# Guarda el resumen en initial_comparison

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

filas = []
for nombre, pipeline in models.items():
    resultados = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="accuracy",
        return_train_score=True,
    )
    # Ojo: cross_validate llama "test_score" al fold de validacion.
    # No es test.csv, que sigue sin abrirse.
    filas.append(
        {
            "modelo": nombre,
            "train_accuracy": resultados["train_score"].mean(),
            "val_accuracy": resultados["test_score"].mean(),
            "val_std": resultados["test_score"].std(),
            "brecha_train_val": resultados["train_score"].mean() - resultados["test_score"].mean(),
        }
    )

initial_comparison = (
    pd.DataFrame(filas).sort_values("val_accuracy", ascending=False).reset_index(drop=True)
)

print("Comparacion inicial (hiperparametros por defecto, 5 folds estratificados):")
print(initial_comparison.round(4).to_string(index=False))


Comparacion inicial (hiperparametros por defecto, 5 folds estratificados):
             modelo  train_accuracy  val_accuracy  val_std  brecha_train_val
logistic_regression          1.0000        0.9791   0.0277            0.0209
                knn          0.9771        0.9507   0.0353            0.0264
      decision_tree          1.0000        0.9022   0.0586            0.0978


## 5. Optimización de hiperparámetros

In [7]:
# TODO: define espacios pequeños de búsqueda para los tres modelos
# TODO: ejecuta GridSearchCV con scoring="accuracy"
# Guarda las búsquedas en searches y el resumen en tuned_comparison

param_grids = {
    # criterion="entropy" es la ganancia de informacion de la clase 6;
    # max_depth y min_samples_leaf son los frenos contra el sobreajuste.
    "decision_tree": {
        "clasificador__criterion": ["gini", "entropy"],
        "clasificador__max_depth": [2, 3, 4, 5, None],
        "clasificador__min_samples_leaf": [1, 2, 5],
    },
    # C es el INVERSO de la fuerza de regularizacion: C pequeno = modelo mas simple.
    "logistic_regression": {
        "clasificador__C": [0.01, 0.1, 1.0, 10.0, 100.0],
    },
    # p=1 distancia Manhattan, p=2 distancia Euclidiana.
    "knn": {
        "clasificador__n_neighbors": [3, 5, 7, 9, 11],
        "clasificador__weights": ["uniform", "distance"],
        "clasificador__p": [1, 2],
    },
}

searches = {}
filas_tuned = []

for nombre, pipeline in models.items():
    busqueda = GridSearchCV(
        pipeline,
        param_grid=param_grids[nombre],
        scoring="accuracy",
        cv=cv,                      # los mismos folds del TODO 3
        return_train_score=True,
        refit=True,
    )
    busqueda.fit(X_train, y_train)
    searches[nombre] = busqueda

    indice = busqueda.best_index_
    cv_res = busqueda.cv_results_
    filas_tuned.append(
        {
            "modelo": nombre,
            "val_accuracy": busqueda.best_score_,
            "val_std": cv_res["std_test_score"][indice],
            "train_accuracy": cv_res["mean_train_score"][indice],
            "brecha_train_val": cv_res["mean_train_score"][indice] - busqueda.best_score_,
            "combinaciones": len(cv_res["params"]),
            "mejores_parametros": busqueda.best_params_,
        }
    )

tuned_comparison = (
    pd.DataFrame(filas_tuned).sort_values("val_accuracy", ascending=False).reset_index(drop=True)
)

print("Comparacion tras optimizar hiperparametros:")
print(tuned_comparison.drop(columns="mejores_parametros").round(4).to_string(index=False))

print("\nMejores hiperparametros por modelo:")
for fila in tuned_comparison.itertuples():
    print(f"  {fila.modelo}: {fila.mejores_parametros}")


Comparacion tras optimizar hiperparametros:
             modelo  val_accuracy  val_std  train_accuracy  brecha_train_val  combinaciones
logistic_regression        0.9791   0.0277          1.0000            0.0209              5
                knn        0.9788   0.0173          0.9806            0.0018             20
      decision_tree        0.9025   0.0762          0.9859            0.0835             30

Mejores hiperparametros por modelo:
  logistic_regression: {'clasificador__C': 1.0}
  knn: {'clasificador__n_neighbors': 7, 'clasificador__p': 1, 'clasificador__weights': 'uniform'}
  decision_tree: {'clasificador__criterion': 'gini', 'clasificador__max_depth': 5, 'clasificador__min_samples_leaf': 2}


## 6. Selección y persistencia del campeón

In [8]:
import sklearn  # solo para dejar la version en los metadatos

# TODO: selecciona el mayor best_score_ de CV
champion_name = max(searches, key=lambda nombre: searches[nombre].best_score_)
champion_search = searches[champion_name]
champion = champion_search.best_estimator_  # pipeline completo, ya reentrenado con refit=True

print("Campeon:", champion_name)
print("Accuracy media de validacion cruzada:", round(champion_search.best_score_, 4))
print("Hiperparametros:", champion_search.best_params_)

# TODO: guarda champion_model.joblib, training_metadata.json y cv_results.csv

# 1) El pipeline completo (preprocesamiento + modelo entrenado)
joblib.dump(champion, ARTIFACTS_DIR / "champion_model.joblib")

# 2) Las 55 combinaciones evaluadas, como evidencia de la seleccion
tablas = []
for nombre, busqueda in searches.items():
    tabla = pd.DataFrame(busqueda.cv_results_)
    tabla.insert(0, "modelo", nombre)
    tablas.append(
        tabla[
            [
                "modelo",
                "params",
                "mean_train_score",
                "mean_test_score",
                "std_test_score",
                "rank_test_score",
            ]
        ]
    )

cv_results = (
    pd.concat(tablas, ignore_index=True)
    .sort_values(["modelo", "rank_test_score"])
    .reset_index(drop=True)
)
cv_results.to_csv(REPORTS_DIR / "cv_results.csv", index=False)

# 3) Metadatos de trazabilidad
# La clave se llama "champion" porque es la que lee el notebook 03.
training_metadata = {
    "champion": champion_name,
    "best_params": champion_search.best_params_,
    "cv_accuracy_mean": float(champion_search.best_score_),
    "cv_accuracy_std": float(champion_search.cv_results_["std_test_score"][champion_search.best_index_]),
    "cv_folds": cv.get_n_splits(),
    "cv_strategy": "StratifiedKFold(n_splits=5, shuffle=True)",
    "scoring": "accuracy",
    "random_state": RANDOM_STATE,
    "target": contract["target"],
    "features": contract["features"],
    "n_train": int(len(X_train)),
    "modelos_comparados": list(searches.keys()),
    "combinaciones_evaluadas": int(len(cv_results)),
    "sklearn_version": sklearn.__version__,
}

with open(ARTIFACTS_DIR / "training_metadata.json", "w", encoding="utf-8") as archivo:
    json.dump(training_metadata, archivo, indent=2, ensure_ascii=False)

print("\ntraining_metadata.json:")
print(json.dumps(training_metadata, indent=2, ensure_ascii=False))


Campeon: logistic_regression
Accuracy media de validacion cruzada: 0.9791
Hiperparametros: {'clasificador__C': 1.0}

training_metadata.json:
{
  "champion": "logistic_regression",
  "best_params": {
    "clasificador__C": 1.0
  },
  "cv_accuracy_mean": 0.9790640394088669,
  "cv_accuracy_std": 0.027713464392187602,
  "cv_folds": 5,
  "cv_strategy": "StratifiedKFold(n_splits=5, shuffle=True)",
  "scoring": "accuracy",
  "random_state": 42,
  "target": "target",
  "features": [
    "alcohol",
    "malic_acid",
    "ash",
    "alcalinity_of_ash",
    "magnesium",
    "total_phenols",
    "flavanoids",
    "nonflavanoid_phenols",
    "proanthocyanins",
    "color_intensity",
    "hue",
    "od280/od315_of_diluted_wines",
    "proline"
  ],
  "n_train": 142,
  "modelos_comparados": [
    "decision_tree",
    "logistic_regression",
    "knn"
  ],
  "combinaciones_evaluadas": 55,
  "sklearn_version": "1.9.1"
}


In [9]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

Challenge 2 completado.


**Justificación de la selección:**

> Se selecciona la **regresión logística** (`C=1.0`) como modelo campeón, con una accuracy media de validación cruzada de **0,9791** sobre 5 folds estratificados.
>
> **Desempeño.** Es el mayor `best_score_` de los tres candidatos, pero la ventaja sobre KNN optimizado (0,9788) es de 0,0003, es decir tres centésimas de punto porcentual. Con 142 registros repartidos en 5 folds —unos 28 por fold, donde un solo acierto vale 3,5 puntos— esa diferencia no representa ni medio ejemplo: ambos modelos están **empatados en la práctica**. El árbol de decisión sí queda claramente atrás, con 0,9025 incluso después de optimizarlo.
>
> **Variabilidad.** KNN es el más estable entre folds (desviación 0,0173), frente a 0,0277 de la regresión logística y 0,0762 del árbol. Este criterio favorece a KNN.
>
> **Sobreajuste.** La brecha entre accuracy de entrenamiento y de validación es de 0,0018 en KNN, 0,0209 en la logística y 0,0835 en el árbol. El árbol memoriza; los otros dos no. Que la logística alcance 100 % en entrenamiento no indica por sí solo sobreajuste: lo que lo indicaría es la brecha, y la suya es de 2 puntos. Ese 100 % refleja que las clases son casi linealmente separables en el espacio de 13 variables, tal como anticipaba el EDA.
>
> **Complejidad e interpretabilidad.** Aquí se rompe el empate. La regresión logística produce un conjunto de coeficientes por clase que puede leerse y discutirse, no necesita conservar los datos de entrenamiento y predice en tiempo constante. KNN no construye modelo alguno: almacena las 142 observaciones y en cada predicción calcula la distancia contra todas, con un costo que crece con el tamaño del conjunto, y su explicación se limita a señalar qué vecinos votaron. Para una etapa de despliegue como la que sigue, la logística es más liviana y más auditable.
>
> **Por qué se descarta el árbol**, pese a ser el más interpretable de los tres: casi 8 puntos de accuracy por debajo y la mayor variabilidad de los tres no se compensan con esa ventaja. Su limitación es estructural — un árbol único corta con umbrales perpendiculares a un eje cada vez, mientras que estas clases se separan mejor mediante fronteras oblicuas que combinan varias variables a la vez.
>
> **Conclusión.** La logística gana por la regla de selección (mayor `best_score_` de validación cruzada) y la decisión se sostiene por simplicidad e interpretabilidad, no por la diferencia de accuracy, que es ruido estadístico. Si en la evaluación final sobre test el desempeño resultara insatisfactorio, KNN con 7 vecinos y distancia Manhattan es la alternativa inmediata — pero esa decisión tendría que tomarse con nuevos datos de validación, nunca reajustando después de mirar el test.

## Checklist

- [x] Todo el procesamiento está dentro de pipelines.
- [x] Comparé tres familias de modelos.
- [x] Optimicé hiperparámetros con CV.
- [x] Elegí el campeón sin usar test.
- [x] Guardé el pipeline completo y sus metadatos.
